In [2]:
import os

base_dir = r"C:\yassmine\reports"

def lister_rapports(base_dir):
    chemins_pdf = []
    for secteur in os.listdir(base_dir):
        secteur_path = os.path.join(base_dir, secteur)
        if os.path.isdir(secteur_path):
            for sous_secteur in os.listdir(secteur_path):
                sous_secteur_path = os.path.join(secteur_path, sous_secteur)
                if os.path.isdir(sous_secteur_path):
                    for fichier in os.listdir(sous_secteur_path):
                        if fichier.endswith(".pdf"):
                            chemins_pdf.append(os.path.join(sous_secteur_path, fichier))
    return chemins_pdf

rapports = lister_rapports(base_dir)


In [3]:
import pdfplumber

def extraire_texte_pdf(pdf_path, max_pages=5):
    texte = ""
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages[:max_pages]:
            texte += page.extract_text() + "\n"
    return texte


In [ ]:
import os
import pdfplumber
import pandas as pd
import json
from openai import OpenAI

# 🔐 Clé API OpenAI (à configurer via variable d'environnement)
api_key = os.getenv("OPENAI_API_KEY")  # Assure-toi que la clé est dans les variables d’environnement
client = OpenAI(api_key=api_key)

# 📁 Répertoires
DOSSIER_TEST = r"C:\yassmine\reports\Renewable Resources & Alternative Energy\UnknownSubsector"
DOSSIER_TEXTES = "rapports_textes"
TAILLE_MIN_KO = 10
LIMITE_FICHIERS = None

# 🐞 Suivi des erreurs
ERREURS_PDF = []
ERREURS_GPT = []

def lister_pdfs_simples(folder, taille_min_ko=10):
    chemins_pdf = []
    for fichier in os.listdir(folder):
        if fichier.lower().endswith(".pdf"):
            chemin = os.path.join(folder, fichier)
            taille = os.path.getsize(chemin) / 1024
            if taille >= taille_min_ko:
                with open(chemin, "rb") as f:
                    if f.read(4) == b"%PDF":
                        chemins_pdf.append(chemin)
                    else:
                        ERREURS_PDF.append({"fichier": chemin, "erreur": "Faux PDF (header invalide)"})
            else:
                ERREURS_PDF.append({"fichier": chemin, "erreur": f"Fichier trop petit ({int(taille)} Ko)"})
    return chemins_pdf

def extraire_texte_pdf(pdf_path, max_pages=5):
    texte = ""
    try:
        with pdfplumber.open(pdf_path) as pdf:
            for page in pdf.pages[:max_pages]:
                contenu = page.extract_text()
                if contenu:
                    texte += contenu + "\n"
    except Exception as e:
        print(f"❌ Erreur lecture PDF : {pdf_path} → {e}")
        ERREURS_PDF.append({"fichier": pdf_path, "erreur": str(e)})
    return texte

def extraire_kpis_gpt(texte, secteur):
    prompt = f"""
Tu es un expert ESG. Voici un extrait d’un rapport d’entreprise du secteur "{secteur}".
Ta tâche est de :
- Lister les *topics ESG* évoqués dans le texte
- En extraire les *KPIs associés*
- Ignorer les généralités

Texte :
{texte}

Retourne la réponse au format JSON strict :
{{
  "topics": ["..."],
  "kpis": ["..."]
}}
"""
    try:
        reponse = client.chat.completions.create(
            model="gpt-4o",
            messages=[{"role": "user", "content": prompt}],
            temperature=0.3
        )
        contenu = reponse.choices[0].message.content.strip()
        print("🧠 Réponse GPT :", contenu[:200])

        if contenu.startswith("```json"):
            contenu = contenu[7:]
        if contenu.endswith("```"):
            contenu = contenu[:-3]

        if not contenu.strip().startswith("{"):
            raise ValueError("⚠️ La réponse n'est pas au format JSON")

        return json.loads(contenu)

    except Exception as e:
        print(f"❌ Erreur GPT : {e}")
        ERREURS_GPT.append({"secteur": secteur, "erreur": str(e)})
        return {"topics": [], "kpis": []}

def traiter_dossier_test():
    fichiers = lister_pdfs_simples(DOSSIER_TEST, TAILLE_MIN_KO)
    if LIMITE_FICHIERS:
        fichiers = fichiers[:LIMITE_FICHIERS]

    print(f"📄 PDF valides trouvés (test) : {len(fichiers)}")
    secteur = "Renewable Resources & Alternative Energy"
    donnees = []

    os.makedirs(DOSSIER_TEXTES, exist_ok=True)

    for i, pdf_path in enumerate(fichiers, 1):
        print(f"🔍 ({i}/{len(fichiers)}) {os.path.basename(pdf_path)}")
        texte = extraire_texte_pdf(pdf_path)
        if not texte.strip():
            print("⚠️ Aucun texte extrait, fichier ignoré.")
            continue

        nom_txt = os.path.basename(pdf_path).replace(".pdf", ".txt")
        with open(os.path.join(DOSSIER_TEXTES, nom_txt), "w", encoding="utf-8") as f:
            f.write(texte)

        resultat = extraire_kpis_gpt(texte, secteur)
        donnees.append({
            "fichier": os.path.basename(pdf_path),
            "topics": ", ".join(resultat.get("topics", [])),
            "kpis": ", ".join(resultat.get("kpis", []))
        })

    pd.DataFrame(donnees).to_excel("resultats_kpis_test.xlsx", index=False)
    pd.DataFrame(ERREURS_PDF).to_csv("erreurs_pdf_test.csv", index=False)
    pd.DataFrame(ERREURS_GPT).to_csv("erreurs_gpt_test.csv", index=False)

    print("✅ Extraction terminée.")
    print(f"📛 Erreurs PDF : {len(ERREURS_PDF)} | 🤖 Erreurs GPT : {len(ERREURS_GPT)}")

if __name__ == "__main__":
    traiter_dossier_test()


📄 PDF valides trouvés (test) : 9
🔍 (1/9) 2W Ecobank SA_2022_Report.pdf
🧠 Réponse GPT : ```json
{
  "topics": [
    "Environmental",
    "Social",
    "Governance",
    "Decarbonization",
    "Renewable Energy",
    "Community Development",
    "Employee Engagement",
    "Sustainability 
🔍 (2/9) AES Brasil_2023_Report.pdf
🧠 Réponse GPT : ```json
{
  "topics": [
    "Operational wind capacity",
    "Renewable energy generation",
    "Partnerships for energy supply",
    "Customer engagement and satisfaction",
    "Deleveraging and fina
🔍 (3/9) Aydem Yenilenebilir Enerji AS_2024_Report.pdf
⚠️ Aucun texte extrait, fichier ignoré.
🔍 (4/9) Bloom Energy Corp_2023_Report.pdf
🧠 Réponse GPT : ```json
{
  "topics": [
    "Decarbonization",
    "Electricity Demand and Supply",
    "Distributed Energy Solutions",
    "Greenhouse Gas Emissions",
    "Energy Storage and Transmission",
    "Dive
🔍 (5/9) Canadian Solar Inc._2022_Report.pdf
🧠 Réponse GPT : ```json
{
  "topics": [
    "Responsible Supply

In [17]:
import pandas as pd
from collections import Counter

# 📥 Charger les résultats précédents
df = pd.read_excel("resultats_kpis_test.xlsx")

# 🧹 Nettoyage et transformation
all_topics = []
all_kpis = []

for _, row in df.iterrows():
    topics = str(row["topics"]).split(",")
    kpis = str(row["kpis"]).split(",")

    # Nettoyage : enlever les espaces et les valeurs vides
    topics = [t.strip() for t in topics if t.strip()]
    kpis = [k.strip() for k in kpis if k.strip()]

    all_topics.extend(topics)
    all_kpis.extend(kpis)

# 🔢 Comptage
topic_counts = Counter(all_topics)
kpi_counts = Counter(all_kpis)

# 🧠 Conversion en DataFrame
df_topics = pd.DataFrame(topic_counts.items(), columns=["Element", "Score"])
df_topics["Type"] = "Topic"

df_kpis = pd.DataFrame(kpi_counts.items(), columns=["Element", "Score"])
df_kpis["Type"] = "KPI"

# 🔀 Fusion finale
df_final = pd.concat([df_topics, df_kpis], ignore_index=True)
df_final = df_final.sort_values(by="Score", ascending=False)

# 💾 Export
df_final.to_excel("scoring_topics_kpis.xlsx", index=False)

print("✅ Scoring terminé → fichier : scoring_topics_kpis.xlsx")


✅ Scoring terminé → fichier : scoring_topics_kpis.xlsx


In [19]:
import pandas as pd
from collections import Counter

# 📥 Charger les résultats précédents
df = pd.read_excel("scoring_topics_kpis.xlsx")

In [20]:
df

,Element,Score,Type
0,Environmental,2,Topic
1,Governance,2,Topic
2,Decarbonization,2,Topic
3,Social,2,Topic
4,Occupational Health and Safety,2,Topic
...,...,...,...
191,Risk Management,1,Topic
192,Green Production,1,Topic
193,Energy Management,1,Topic
194,Emission Management,1,Topic


In [28]:
import pandas as pd

# 📥 Charger les données
df = pd.read_excel("scoring_topics_kpis.xlsx")

# ✅ Nettoyer les noms de colonnes
df.columns = [col.lower().strip() for col in df.columns]
print("📋 Colonnes détectées :", df.columns.tolist())

# 🛡️ Vérification des colonnes nécessaires
if not {"element", "type"}.issubset(df.columns):
    raise ValueError("❌ Les colonnes 'element' et 'type' sont requises dans le fichier.")

# 🎯 Séparer topics et KPIs
df_topics = df[df["type"].str.lower() == "topic"]
df_kpis = df[df["type"].str.lower() == "kpi"]

# 🔢 Compter les fréquences
frequence_topics = df_topics["element"].value_counts().reset_index()
frequence_topics.columns = ["Topic", "Fréquence"]

frequence_kpis = df_kpis["element"].value_counts().reset_index()
frequence_kpis.columns = ["KPI", "Fréquence"]

# 💾 Exporter vers Excel
frequence_topics.to_excel("frequence_topics.xlsx", index=False)
frequence_kpis.to_excel("frequence_kpis.xlsx", index=False)

# 👀 Affichage console
print("\n📊 Top 5 des Topics :")
print(frequence_topics.head())

print("\n📊 Top 5 des KPIs :")
print(frequence_kpis.head())

print("\n✅ Fichiers exportés : frequence_topics.xlsx et frequence_kpis.xlsx")


📋 Colonnes détectées : ['element', 'score', 'type']

📊 Top 5 des Topics :
                                               Topic  Fréquence
0                                      Environmental          1
1  Greenhouse Gas Emissions & Energy Resource Pla...          1
2                             Resource Reutilization          1
3                               Community Engagement          1
4  Environmental Investment and Regulatory Compli...          1

📊 Top 5 des KPIs :
                                                 KPI  Fréquence
0                       ESG Management and Oversight          1
1  EcoVadis Global Supply Chain Evaluation - Silv...          1
2                     TCFD alignment and disclosures          1
3                       Emissions and energy metrics          1
4                                Sustainability KPIs          1

✅ Fichiers exportés : frequence_topics.xlsx et frequence_kpis.xlsx


In [ ]:
import pandas as pd
from openai import OpenAI
import os

# 🔐 Clé API chargée depuis une variable d’environnement
api_key = os.getenv("OPENAI_API_KEY")
client = OpenAI(api_key=api_key)

# 📥 Charger les KPIs
df_kpis = pd.read_excel("frequence_kpis.xlsx")

# ✅ Nettoyage des données
kpis = df_kpis["KPI"].dropna().unique().tolist()

# 🧠 Générer une question à partir du KPI
def generer_question(kpi):
    prompt = f"""
Tu es un expert en audit RSE/ESG.
Génère une question simple, claire et précise qu’un analyste poserait pour obtenir la valeur du KPI suivant :
→ "{kpi}"

Réponds uniquement par la question.
    """
    try:
        completion = client.chat.completions.create(
            model="gpt-4o",
            messages=[{"role": "user", "content": prompt}],
            temperature=0
        )
        return completion.choices[0].message.content.strip()
    except Exception as e:
        print(f"❌ Erreur GPT pour le KPI '{kpi}' :", e)
        return "Erreur GPT"

# 🔄 Générer les questions pour chaque KPI
questions = [{"KPI": kpi, "Question": generer_question(kpi)} for kpi in kpis]

# 💾 Sauvegarde dans un fichier Excel
df_questions = pd.DataFrame(questions)
df_questions.to_excel("questions_par_kpi.xlsx", index=False)
print("✅ Fichier 'questions_par_kpi.xlsx' généré.")


✅ Fichier 'questions_par_kpi.xlsx' généré.
